# 3 · A larger LLM, prompted only

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emrtunali/yetkin/blob/main/notebooks/03_prompting.ipynb)

No training: the same instructions as the QLoRA model, answered by a 7B instruction model in 4-bit, **zero-shot**
and with **five worked examples** from the training set. This is what "just ask an LLM" gets you.

About 5–6 hours on a T4 (20 minutes per 500 sentences zero-shot, longer with five examples in the prompt), on a
*High-RAM* runtime: the 7B model does not load in the standard 12.7 GB. Answers are saved in blocks of 500
sentences, so after a disconnect *Run all* continues from the last block.

In [ ]:
# Runtime → Change runtime type → T4 GPU, then Runtime → Run all.
!pip -q install "yetkin[train,qlora,track] @ git+https://github.com/emrtunali/yetkin.git"

import json, os, time
import torch
from google.colab import drive, userdata

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/yetkin"          # results survive a disconnect here
os.makedirs(f"{OUT}/predictions", exist_ok=True)

def secret(name):
    """Optional keys from the 🔑 Secrets panel: HF_TOKEN (to publish models), WANDB_API_KEY (to log runs)."""
    try:
        return userdata.get(name)
    except Exception:
        return None

In [ ]:
from yetkin.data import load, chunks
train = [c for s in load("train") for c in chunks(s, 128)]
dev, test = load("dev"), load("test")
print(f"train {len(train):,} sentences · dev {len(dev):,} · test {len(test):,}")

In [ ]:
import random
BASE = "Qwen/Qwen2.5-7B-Instruct"
both = [s for s in train if s.of("skill") and s.of("knowledge") and len(s.tokens) <= 40]
SHOTS = random.Random(0).sample(both, 5)
for s in SHOTS: print("·", " ".join(s.tokens))

In [ ]:
from yetkin import llm
from yetkin.io import save_predictions, load_predictions
from yetkin.data import Span
model, tok = llm.load(BASE, four_bit=True)

def run(tag, shots):
    final = f"{OUT}/predictions/prompt-{tag}.jsonl"
    if os.path.exists(final):
        print("done already:", final); return
    parts, t0 = [], time.time()
    for b in range(0, len(test), 500):
        part = f"{OUT}/predictions/.prompt-{tag}-{b:05d}.jsonl"
        block = test[b:b + 500]
        if not os.path.exists(part):
            pred, raw = llm.extract(model, tok, block, shots=shots, batch_size=16)
            save_predictions(part, block, pred, raw=raw)
            print(f"{tag}: {b + len(block)}/{len(test)}")
        parts.append(part)
    pred = [p for part in parts for p in load_predictions(part)[0]]
    per_1k = (time.time() - t0) / len(test) * 1000
    label = "zero-shot" if not shots else f"{len(shots)}-shot"
    save_predictions(final, test, pred, meta={"name": f"Prompted · {BASE.split('/')[-1]} · {label}", "system": f"Prompted {label}",
                                               "base": BASE, "shots": len(shots), "seconds_per_1k_sentences": round(per_1k, 1)})
    for p in parts: os.remove(p)

run("zero-shot", [])
run("5-shot", SHOTS)